# 🩺 Análise Multimodal de Áudio Clínico: Eventos Não-Verbais, Emoções na Voz e Discordância no Discurso

Este notebook implementa um pipeline completo para análise de consultas médicas em áudio (interação **Paciente–Médico**), combinando modelos locais de áudio e inteligência artificial generativa:

1. **Detecção de Sons Não-Verbais Medicamente Relevantes**: Identificação temporal de **tosse**, **espirro**, **limpeza de garganta (pigarro)**, **suspiro**, **gemido de dor** e **respiração ofegante** utilizando o modelo **Audio Spectrogram Transformer (AST)** pré-treinado no *AudioSet* (`MIT/ast-finetuned-audioset-10-10-0.4593`).
2. **Transcrição Segmentada com Whisper**: Uso do modelo local **OpenAI Whisper** para transcrever a consulta com *timestamps* precisos por segmento de fala.
3. **Reconhecimento Local de Emoções por Tom de Voz (SER)**: Classificação acústica das emoções em cada segmento de fala utilizando um modelo local baseado em **Wav2Vec2** (`ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition`), complementado pela extração de métricas prosódicas via `librosa` (frequência fundamental $F_0$, energia RMS e variabilidade vocal).
4. **Análise Semântica e Comparação de Discordâncias via OpenAI API**: Uso do modelo da OpenAI (`gpt-4o-mini` / `gpt-4o`) para avaliar a emoção transmitida pelas palavras do discurso transcrito, identificar o interlocutor (*Paciente* ou *Médico*) e detectar **discordâncias (incongruências) emocionais** entre o que é dito verbalmente e o que o tom de voz revela.
5. **Relatório Clínico de Discordâncias e Linha do Tempo Multimodal**: Consolidação automática em relatório estruturado (`.md` e `.csv`) e visualizações gráficas.

## 1. Instalação e Importação de Bibliotecas
Descomente a célula abaixo caso precise instalar os pacotes no ambiente atual.

### Instalação de dependências para Diarização de Falantes
Para utilizar o `pyannote.audio`, precisamos instalar a biblioteca e suas dependências adicionais de processamento.

In [58]:
!pip install -q pyannote.audio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.4 which is incompatible.


In [59]:
!pip install -q openai-whisper transformers torch torchaudio librosa soundfile openai pydantic pandas numpy matplotlib seaborn gTTS pydub python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytorch-metric-learning 3.0.2 requires numpy>=2.5.3, but you have numpy 2.4.6 which is incompatible.


In [60]:
import os
import json
import warnings
from pathlib import Path
from typing import List, Optional, Dict, Any, Literal

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
import whisper
from transformers import pipeline
from pydantic import BaseModel, Field
from google.colab import userdata
from openai import OpenAI
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, Markdown, Audio

warnings.filterwarnings('ignore')
load_dotenv()

DEVICE = 0 if torch.cuda.is_available() else -1
TORCH_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo de execução selecionado: {TORCH_DEVICE.upper()}')

Dispositivo de execução selecionado: CUDA


## 2. Configuração da API da OpenAI e Parâmetros do Projeto
Defina sua chave `OPENAI_API_KEY` no arquivo `.env` ou diretamente na variável de ambiente abaixo.

In [61]:
# Caso não esteja usando arquivo .env, descomente e insira sua chave abaixo:
# os.environ["OPENAI_API_KEY"] = "sk-..."

OPENAI_MODEL = "gpt-4o-mini"  # Pode ser alterado para "gpt-4o"
WHISPER_MODEL_SIZE = "base"   # Opções: "tiny", "base", "small", "medium", "large"
SAMPLE_RATE = 16000

try:
    api_key = userdata.get("OPENAI_API_KEY")
    client = OpenAI(api_key=api_key)
except Exception as e:
    print("Chave 'OPENAI_API_KEY' não encontrada no Colab Secrets. Configure para usar a análise por LLM.")
    client = None

## 3. Obtenção ou Geração de Áudio Clínico de Exemplo
Se você já possui um arquivo de áudio de consulta médica (`.wav` ou `.mp3`), basta apontar o caminho em `AUDIO_PATH`.

Para permitir que este notebook seja executado imediatamente de ponta a ponta, a função abaixo gera automaticamente um áudio simulado de consulta clínica caso o arquivo não exista, intercalando:
- Falas entre **Médico** e **Paciente** (incluindo frases onde o paciente minimiza os sintomas verbalmente, como *"Está tudo ótimo doutor, não estou sentindo nada preocupante"*, para testar a análise de discordância emocional);
- Eventos acústicos não-verbais inseridos entre as falas (**tosse**, **limpeza de garganta/pigarro** e **espirro**).

In [62]:
# A verificação de áudio foi movida para o bloco de execução final.

## 4. Módulo 1: Reconhecimento de Sons Não-Verbais Medicamente Relevantes
Utilizamos o modelo **AST (Audio Spectrogram Transformer)** pré-treinado no *AudioSet* (`MIT/ast-finetuned-audioset-10-10-0.4593`) para varrer o áudio com janelas deslizantes e detectar eventos não-verbais relevantes para o contexto clínico:
- **Tosse** (`Cough`)
- **Limpeza de garganta / Pigarro** (`Throat clearing`)
- **Espirro** (`Sneeze`)
- **Suspiro / Ofego / Falta de ar** (`Sigh`, `Pant`, `Gasp`, `Wheeze`, `Breathing`)
- **Fungada / Coriza** (`Sniff`, `Snort`)
- **Choro / Gemido de dor** (`Crying, sobbing`, `Groan`, `Grunt`, `Whimper`)

In [63]:
# Mapeamento de rótulos do AudioSet para categorias clínicas em Português
CLINICAL_SOUND_MAP: Dict[str, str] = {
    "Cough": "Tosse",
    "Throat clearing": "Limpeza de Garganta (Pigarro)",
    "Sneeze": "Espirro",
    "Sniff": "Fungada / Coriza",
    "Sigh": "Suspiro",
    "Gasp": "Arquejo / Falta de Ar",
    "Pant": "Respiração Ofegante",
    "Wheeze": "Sibilo / Chiado Respiratório",
    "Groan": "Gemido de Dor/Desconforto",
    "Grunt": "Grunhido de Esforço/Dor",
    "Crying, sobbing": "Choro / Soluço",
    "Whimper": "Queixume / Choro Contido",
    "Snort": "Ronco / Obstrução Nasal",
}

print("Carregando modelo AST para detecção de eventos acústicos não-verbais...")
ast_classifier = pipeline(
    "audio-classification",
    model="MIT/ast-finetuned-audioset-10-10-0.4593",
    device=DEVICE
)

Carregando modelo AST para detecção de eventos acústicos não-verbais...


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

In [64]:
def detect_clinical_non_verbal_sounds(
    audio_path: Path,
    window_sec: float = 1.2,
    hop_sec: float = 0.6,
    score_threshold: float = 0.12,
    rms_min_threshold: float = 0.015
) -> pd.DataFrame:
    """
    Analisa o áudio em janelas deslizantes usando o modelo AST (AudioSet)
    combinado a descritores espectrais para identificar sons não-verbais clínicos.
    """
    y, sr = librosa.load(str(audio_path), sr=SAMPLE_RATE, mono=True)
    win_len = int(window_sec * sr)
    hop_len = int(hop_sec * sr)

    raw_detections = []

    for start_sample in range(0, max(1, len(y) - win_len + 1), hop_len):
        end_sample = min(len(y), start_sample + win_len)
        segment = y[start_sample:end_sample]
        start_t = round(start_sample / sr, 2)
        end_t = round(end_sample / sr, 2)

        rms = float(np.sqrt(np.mean(segment ** 2)))
        if rms < rms_min_threshold:
            continue

        preds = ast_classifier({"raw": segment, "sampling_rate": sr}, top_k=8)
        speech_score = sum(p["score"] for p in preds if p["label"] in ("Speech", "Conversation", "Narration, monologue"))

        for p in preds:
            label = p["label"]
            score = float(p["score"])
            if label in CLINICAL_SOUND_MAP and score >= score_threshold:
                raw_detections.append({
                    "inicio_s": start_t,
                    "fim_s": end_t,
                    "evento_en": label,
                    "evento_clinico": CLINICAL_SOUND_MAP[label],
                    "confianca": round(score, 3),
                    "energia_rms": round(rms, 4),
                    "prob_fala_simultanea": round(speech_score, 3)
                })
                break

    # Consolidação de janelas adjacentes do mesmo evento
    if not raw_detections:
        return pd.DataFrame(columns=["inicio_s", "fim_s", "evento_clinico", "evento_en", "confianca", "energia_rms"])

    merged = [raw_detections[0]]
    for det in raw_detections[1:]:
        prev = merged[-1]
        if det["evento_clinico"] == prev["evento_clinico"] and det["inicio_s"] <= prev["fim_s"] + 0.35:
            prev["fim_s"] = max(prev["fim_s"], det["fim_s"])
            prev["confianca"] = round(max(prev["confianca"], det["confianca"]), 3)
            prev["energia_rms"] = round(max(prev["energia_rms"], det["energia_rms"]), 4)
        else:
            merged.append(det)

    return pd.DataFrame(merged)

## 5. Módulo 2: Transcrição do Áudio com OpenAI Whisper (Local)
Utilizamos o modelo **OpenAI Whisper** localmente para realizar a transcrição completa do diálogo médico-paciente, preservando o tempo de início (`start`) e fim (`end`) de cada segmento falado.

In [65]:
print(f"Carregando modelo Whisper ({WHISPER_MODEL_SIZE})...")
whisper_model = whisper.load_model(WHISPER_MODEL_SIZE, device=TORCH_DEVICE)

def transcribe_clinical_audio(audio_path: Path, language: str = "en") -> List[Dict[str, Any]]:
    """Transcreve o áudio com Whisper retornando segmentos temporais detalhados."""
    result = whisper_model.transcribe(
        str(audio_path),
        language=language,
        verbose=False,
        fp16=torch.cuda.is_available()
    )
    segments = []
    for idx, seg in enumerate(result.get("segments", [])):
        text = seg.get("text", "").strip()
        if text:
            segments.append({
                "segment_id": idx + 1,
                "inicio_s": round(float(seg["start"]), 2),
                "fim_s": round(float(seg["end"]), 2),
                "transcricao": text
            })
    return segments

Carregando modelo Whisper (base)...


## 5.1. Diarização e Identificação de Falantes com Pyannote

Nesta etapa, carregamos o pipeline de Diarização do Pyannote para identificar os turnos de fala dos dois participantes (*Doctor* e *Patient*). Para usar os modelos do Pyannote, é necessário aceitar os termos de uso no Hugging Face (modelos `pyannote/speaker-diarization-3.1` ou `pyannote/speaker-diarization`) e obter um Token de Acesso (HF_TOKEN) configurado no segredo do Colab.

In [66]:
from pyannote.audio import Pipeline

print("Carregando o pipeline de diarização oficial pyannote/speaker-diarization-community-1...")
HF_TOKEN = userdata.get("HF_TOKEN")

# Carrega exatamente o pipeline solicitado pelo usuário
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN
)

if torch.cuda.is_available() and diarization_pipeline is not None:
    diarization_pipeline.to(torch.device("cuda"))

print("Pipeline pyannote/speaker-diarization-community-1 carregado com sucesso!")

Carregando o pipeline de diarização oficial pyannote/speaker-diarization-community-1...
Pipeline pyannote/speaker-diarization-community-1 carregado com sucesso!


In [67]:
from pathlib import Path
from typing import List, Dict, Any

def run_diarization(audio_path: Path) -> List[Dict[str, Any]]:
    """
    Executa a diarização e retorna uma lista de intervalos de tempo com seus respectivos speakers.
    Lança um erro caso o pipeline não tenha sido inicializado corretamente.
    """
    if diarization_pipeline is None:
        raise RuntimeError("O pipeline de diarização do Pyannote não foi carregado. Verifique suas credenciais e o token HF_TOKEN.")

    diarization = diarization_pipeline(str(audio_path), num_speakers=2)
    segments = []
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        segments.append({
            "inicio_s": round(turn.start, 2),
            "fim_s": round(turn.end, 2),
            "speaker_raw": speaker
        })
    return segments

In [68]:
# Executa a diarização acústica real no áudio completo usando o modelo community-1

def run_real_diarization(audio_path: Path) -> List[Dict[str, Any]]:
    output = diarization_pipeline(str(audio_path))

    segments = []

    # Siga estritamente a especificação oficial do modelo 'speaker-diarization-community-1'
    for turn, speaker in output.speaker_diarization:
        segments.append({
            "inicio_s": round(turn.start, 2),
            "fim_s": round(turn.end, 2),
            "speaker_raw": speaker
        })

    return segments#

## 5.2. Filtrando Apenas os Segmentos do Paciente
Utilizando a regra de negócio estabelecida de que um dos interlocutores se identifica como o Médico/Robô Doutor (que inicia a conversa com perguntas no script de simulação), mapeamos quem é o **Paciente** e filtramos o pipeline para conter unicamente esses trechos no processamento acústico de sentimentos e discordância verbal.

In [69]:
# Identificação acústica do Paciente baseada no cruzamento de tempos robusto entre Whisper e Pyannote

def get_patient_speaker_id_acoustically(diarized_segments: List[Dict[str, Any]], transcription_segments: List[Dict[str, Any]]) -> str:
    """
    Localiza o falante que se apresenta como médico/robô e define o outro como Paciente,
    usando interseção temporal e análise de palavras-chave.
    """
    robot_speaker = None

    # Palavras-chave indicando o médico/robô
    keywords = ["doctor", "robot", "doutor", "bom dia", "sentido", "consulta"]

    for seg in transcription_segments:
        texto = seg["transcricao"].lower()
        if any(kw in texto for kw in keywords):
            seg_start = seg["inicio_s"]
            seg_end = seg["fim_s"]

            # Encontra qual speaker tem maior sobreposição temporal com este segmento
            best_speaker = None
            max_overlap = 0.0
            for turn in diarized_segments:
                # Calcula a sobreposição (interseção) de intervalos
                overlap_start = max(seg_start, turn["inicio_s"])
                overlap_end = min(seg_end, turn["fim_s"])
                overlap = max(0.0, overlap_end - overlap_start)

                if overlap > max_overlap:
                    max_overlap = overlap
                    best_speaker = turn["speaker_raw"]

            if best_speaker:
                robot_speaker = best_speaker
                break

    all_speakers = list(set(turn["speaker_raw"] for turn in diarized_segments))

    if robot_speaker and len(all_speakers) >= 2:
        patient_speaker = [sp for sp in all_speakers if sp != robot_speaker][0]
    else:
        # Caso não consiga diferenciar pelo texto, assumimos o speaker que mais fala ou o SPEAKER_01 como padrão
        speaker_counts = {}
        for turn in diarized_segments:
            dur = turn["fim_s"] - turn["inicio_s"]
            speaker_counts[turn["speaker_raw"]] = speaker_counts.get(turn["speaker_raw"], 0.0) + dur

        # Paciente costuma falar menos ou ser o segundo speaker ativo
        sorted_speakers = sorted(speaker_counts.items(), key=lambda x: x[1])
        if sorted_speakers:
            patient_speaker = sorted_speakers[0][0]
        else:
            patient_speaker = "SPEAKER_01"

    return patient_speaker

def filter_patient_segments(diarized_turns: List[Dict[str, Any]], transcribed_segments: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    patient_speaker_id = get_patient_speaker_id_acoustically(diarized_turns, transcribed_segments)
    patient_segments = []
    for seg in transcribed_segments:
        seg_start = seg["inicio_s"]
        seg_end = seg["fim_s"]

        best_speaker = None
        max_overlap = 0.0

        for turn in diarized_turns:
            overlap_start = max(seg_start, turn["inicio_s"])
            overlap_end = min(seg_end, turn["fim_s"])
            overlap = max(0.0, overlap_end - overlap_start)

            if overlap > max_overlap:
                max_overlap = overlap
                best_speaker = turn["speaker_raw"]

        if best_speaker == patient_speaker_id:
            patient_segments.append(seg)
    return patient_segments

## 6. Módulo 3: Reconhecimento Local de Emoções por Tom de Voz (SER)
Para cada segmento de fala transcrito pelo Whisper, recortamos o trecho correspondente do áudio original e aplicamos:
1. **Modelo Local de Speech Emotion Recognition (SER)**: `ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition` (arquitetura Wav2Vec2 XLSR ajustada para reconhecimento de emoções vocais como *tristeza, medo/ansiedade, raiva, calma, felicidade, desgosto, surpresa e neutro*).
2. **Extração de Características Prosódicas (`librosa`)**: Frequência fundamental média ($F_0$ / Pitch), desvio padrão do Pitch (instabilidade/tremor vocal), energia RMS e taxa de cruzamento por zero (ZCR), que auxiliam na interpretação acústica da emoção.

In [70]:
SER_MODEL_NAME = "ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition"

EMOTION_TRANSLATION_MAP: Dict[str, str] = {
    "angry": "Raiva / Irritação",
    "calm": "Calma",
    "disgust": "Desconforto / Aversão",
    "fearful": "Medo / Ansiedade",
    "happy": "Alegria / Entusiasmo",
    "neutral": "Neutro",
    "sad": "Tristeza / Abatimento",
    "surprised": "Surpresa / Espanto",
    # Compatibilidade caso outro modelo Wav2Vec2 (ex: superb-er) seja usado
    "ang": "Raiva / Irritação",
    "hap": "Alegria / Entusiasmo",
    "neu": "Neutro",
    "sad": "Tristeza / Abatimento"
}

print(f"Carregando modelo local de emoções por tom de voz: {SER_MODEL_NAME}...")
ser_pipeline = pipeline(
    "audio-classification",
    model=SER_MODEL_NAME,
    device=DEVICE
)

Carregando modelo local de emoções por tom de voz: ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition...


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition
Key                      | Status     | 
-------------------------+------------+-
classifier.output.bias   | UNEXPECTED | 
classifier.dense.bias    | UNEXPECTED | 
classifier.dense.weight  | UNEXPECTED | 
classifier.output.weight | UNEXPECTED | 
classifier.weight        | MISSING    | 
classifier.bias          | MISSING    | 
projector.bias           | MISSING    | 
projector.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [71]:
def extract_prosodic_features(segment_audio: np.ndarray, sr: int = SAMPLE_RATE) -> Dict[str, float]:
    """Extrai métricas prosódicas (Pitch F0, variação de Pitch e Energia RMS) de um trecho de voz."""
    if len(segment_audio) < int(0.15 * sr):
        return {"pitch_medio_hz": 0.0, "pitch_std_hz": 0.0, "energia_rms": 0.0}

    rms = float(np.sqrt(np.mean(segment_audio ** 2)))
    f0, voiced_flag, _ = librosa.pyin(
        segment_audio,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C6"),
        sr=sr
    )
    valid_f0 = f0[~np.isnan(f0)] if f0 is not None else np.array([])
    pitch_mean = float(np.mean(valid_f0)) if len(valid_f0) > 0 else 0.0
    pitch_std = float(np.std(valid_f0)) if len(valid_f0) > 0 else 0.0

    return {
        "pitch_medio_hz": round(pitch_mean, 1),
        "pitch_std_hz": round(pitch_std, 1),
        "energia_rms": round(rms, 4)
    }


def analyze_voice_emotions(
    audio_path: Path,
    segments: List[Dict[str, Any]],
    non_verbal_df: pd.DataFrame
) -> List[Dict[str, Any]]:
    """
    Executa o modelo local de reconhecimento de emoções por tom de voz em cada segmento
    transcrito e associa eventos não-verbais próximos (+/- 2.0 segundos).
    """
    y, sr = librosa.load(str(audio_path), sr=SAMPLE_RATE, mono=True)
    enriched_segments = []

    for seg in segments:
        start_sample = max(0, int(seg["inicio_s"] * sr))
        end_sample = min(len(y), int(seg["fim_s"] * sr))
        clip = y[start_sample:end_sample]

        if len(clip) < int(0.2 * sr):
            clip = np.pad(clip, (0, int(0.2 * sr) - len(clip)))

        ser_preds = ser_pipeline({"raw": clip, "sampling_rate": sr}, top_k=3)
        top1 = ser_preds[0]
        top2 = ser_preds[1] if len(ser_preds) > 1 else {"label": "neutral", "score": 0.0}

        emo_primaria = EMOTION_TRANSLATION_MAP.get(top1["label"], top1["label"])
        emo_secundaria = EMOTION_TRANSLATION_MAP.get(top2["label"], top2["label"])
        prosody = extract_prosodic_features(clip, sr=sr)

        # Busca sons não-verbais que ocorreram próximos ao segmento (janela de 2.5s antes/depois)
        nearby_events = []
        if not non_verbal_df.empty:
            mask = (
                (non_verbal_df["fim_s"] >= seg["inicio_s"] - 2.5)
                & (non_verbal_df["inicio_s"] <= seg["fim_s"] + 2.5)
            )
            nearby_events = non_verbal_df.loc[mask, "evento_clinico"].unique().tolist()

        enriched_segments.append({
            **seg,
            "emocao_voz_primaria": emo_primaria,
            "confianca_voz_primaria": round(float(top1["score"]), 3),
            "emocao_voz_secundaria": emo_secundaria,
            "confianca_voz_secundaria": round(float(top2["score"]), 3),
            **prosody,
            "eventos_nao_verbais_proximos": nearby_events
        })

    return enriched_segments

## 7. Módulo 4: Análise das Emoções no Discurso Transcrito (OpenAI API) e Comparação Local
Nesta etapa, enviamos os segmentos transcritos para a **API da OpenAI** usando *Structured Outputs* (`pydantic`) **apenas** para classificar a emoção semântica declarada no texto.

A detecção de **discordância emocional** será feita localmente via código Python, comparando a emoção identificada no texto com a emoção acústica detectada pelo Wav2Vec2 e a presença de eventos físicos (tosse, etc).

In [72]:
class SegmentTextEmotion(BaseModel):
    segment_id: int = Field(description="ID numérico do segmento analisado.")
    emocao_discurso_texto: str = Field(description="Emoção transmitida estritamente pelo conteúdo verbal/semântico da frase.")

class ConsultationTextEmotions(BaseModel):
    segmentos_analisados: List[SegmentTextEmotion]

In [73]:
def analyze_text_emotion_with_openai(
    voice_segments: List[Dict[str, Any]],
    model_name: str = OPENAI_MODEL
) -> ConsultationTextEmotions:
    """Utiliza a API da OpenAI apenas para extrair a emoção do texto transcrito."""
    payload = [{"segment_id": s["segment_id"], "transcricao": s["transcricao"]} for s in voice_segments]

    system_prompt = (
        "Você é um assistente de extração de características. Identifique a emoção predominante "
        "estritamente pelo conteúdo verbal de cada segmento. Use termos simples como: "
        "'Alegria', 'Calma', 'Neutro', 'Tristeza', 'Raiva', 'Medo', 'Ansiedade', 'Desconforto'."
    )

    if client is None:
         return ConsultationTextEmotions(segmentos_analisados=[
             SegmentTextEmotion(segment_id=s["segment_id"], emocao_discurso_texto="Neutro")
             for s in voice_segments
         ])

    response = client.beta.chat.completions.parse(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Analise o texto:\n{json.dumps(payload, ensure_ascii=False, indent=2)}"}
        ],
        response_format=ConsultationTextEmotions,
        temperature=0.0
    )
    return response.choices[0].message.parsed

def calculate_discordance_locally(segments: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Compara text_emotion vs voice_emotion e non_verbal_events programaticamente."""
    results = []
    positive_emotions = ["alegria", "calma", "neutro", "tranquilidade", "otimismo", "bem-estar", "positivo"]
    negative_emotions = ["tristeza", "raiva", "medo", "ansiedade", "desconforto", "aversão", "abatimento", "irritação", "negativo"]

    for seg in segments:
        texto_emo = seg.get("emocao_discurso_texto", "").lower()
        voz_emo = seg.get("emocao_voz_primaria", "").lower()

        is_text_positive = any(p in texto_emo for p in positive_emotions)
        is_voice_negative = any(n in voz_emo for n in negative_emotions)

        is_text_negative = any(n in texto_emo for n in negative_emotions)
        is_voice_positive = any(p in voz_emo for p in positive_emotions)

        discordancia = False
        nivel = "Nenhuma"
        explicacao = "Convergência entre texto e voz."

        # Regra 1: Text Positivo x Voz Negativa (ou vice-versa)
        if (is_text_positive and is_voice_negative) or (is_text_negative and is_voice_positive):
            discordancia = True
            nivel = "Alta" if seg.get("confianca_voz_primaria", 0) > 0.6 else "Moderada"
            explicacao = f"Divergência direta: Texto indicou '{seg['emocao_discurso_texto']}' mas Voz indicou '{seg['emocao_voz_primaria']}'."
        # Regra 2: Sinais Físicos com Texto Positivo
        elif seg.get("eventos_nao_verbais_proximos") and is_text_positive:
            discordancia = True
            nivel = "Leve"
            explicacao = f"Sinais físicos ({', '.join(seg['eventos_nao_verbais_proximos'])}) contradizem o relato textual positivo/neutro."

        seg_copy = seg.copy()
        seg_copy["ha_discordancia"] = discordancia
        seg_copy["nivel_discordancia"] = nivel
        seg_copy["explicacao_clinica"] = explicacao
        results.append(seg_copy)
    return results

def run_text_and_discordance_analysis(segments_with_voice_emotion: List[Dict[str, Any]]) -> pd.DataFrame:
    # 1. Obtém as emoções do texto via OpenAI
    text_emotions = analyze_text_emotion_with_openai(segments_with_voice_emotion)
    df_text_emotions = pd.DataFrame([s.model_dump() for s in text_emotions.segmentos_analisados])

    # 2. Mescla os dados para uso da lógica local
    merged_segments = []
    for seg in segments_with_voice_emotion:
        text_emo_row = df_text_emotions[df_text_emotions["segment_id"] == seg["segment_id"]]
        text_emo = text_emo_row.iloc[0]["emocao_discurso_texto"] if not text_emo_row.empty else "Neutro"
        new_seg = seg.copy()
        new_seg["emocao_discurso_texto"] = text_emo
        new_seg["interlocutor"] = "Paciente"
        merged_segments.append(new_seg)

    # 3. Calcula discordâncias com lógica simples e determinística
    final_segments = calculate_discordance_locally(merged_segments)
    return pd.DataFrame(final_segments)

## 9. Módulo 6: Geração do Relatório de Discordâncias Emocionais
Por fim, geramos um relatório clínico estruturado **automaticamente a partir das lógicas programáticas computadas**, sem o uso da inteligência artificial para redação:
- **Resumo Quantitativo da Consulta** (frequência de discordâncias);
- **Tabela de Eventos Acústicos Não-Verbais** (Tosse, Espirro, Limpeza de Garganta e afins);
- **Mapeamento Completo de Discordâncias**;
- **Detalhamento Programático das Discordâncias Identificadas**.

O relatório é exibido formatado no notebook e salvo automaticamente em `relatorio_discordancias_clinicas.md`.

In [76]:
def generate_clinical_discordance_report(
    df_full: pd.DataFrame,
    df_events: pd.DataFrame,
    output_md_path: Path = Path("relatorio_discordancias_clinicas.md")
) -> str:
    """Formata e salva o relatório programático de discordâncias emocionais e eventos não-verbais."""
    total_segs = len(df_full)
    discordant_df = df_full[df_full["ha_discordancia"] == True]
    num_discordant = len(discordant_df)

    lines = [
        "# 📋 Relatório Clínico Analítico: Eventos Não-Verbais e Discordâncias Emocionais",
        "",
        "## 1. Resumo Quantitativo da Consulta",
        f"- **Total de segmentos analisados:** {total_segs}",
        f"- **Total de discordâncias detectadas:** {num_discordant} ({(num_discordant/max(1, total_segs))*100:.1f}%)",
        f"- **Total de eventos não-verbais:** {len(df_events)}",
        "",
        "## 2. Sons Não-Verbais Detectados"
    ]

    if df_events.empty:
        lines.append("_Nenhum evento não-verbal relevante detectado._\n")
    else:
        lines.append("| Início (s) | Fim (s) | Evento Clínico Detectado | Confiança Acústica |")
        lines.append("| :---: | :---: | :--- | :---: |")
        for _, ev in df_events.iterrows():
            lines.append(
                f"| {ev['inicio_s']:.2f}s | {ev['fim_s']:.2f}s | **{ev['evento_clinico']}** | "
                f"{ev['confianca']*100:.1f}% |"
            )
        lines.append("")

    lines.extend([
        "## 3. Comparativo Geral: Texto vs Voz",
        "| Seg # | Tempo | Transcrição | Emoção Texto | Emoção Voz | Sons Próximos | Discordância |",
        "| :---: | :---: | :--- | :--- | :--- | :--- | :---: |"
    ])

    for _, row in df_full.iterrows():
        ev_str = ", ".join(row["eventos_nao_verbais_proximos"]) if row["eventos_nao_verbais_proximos"] else "—"
        disc_badge = f"⚠️ **{row['nivel_discordancia']}**" if row["ha_discordancia"] else "✅ Nenhuma"
        lines.append(
            f"| {row['segment_id']} | {row['inicio_s']:.1f}s–{row['fim_s']:.1f}s | "
            f"\"{row['transcricao']}\" | {row['emocao_discurso_texto']} | "
            f"{row['emocao_voz_primaria']} | "
            f"{ev_str} | {disc_badge} |"
        )

    lines.extend([
        "",
        "## 4. Detalhamento Clínico Programático",
    ])

    if discordant_df.empty:
        lines.append("_Não foram identificadas discordâncias significativas pelo sistema._\n")
    else:
        for _, row in discordant_df.iterrows():
            lines.extend([
                f"### 🔍 Segmento #{row['segment_id']} ({row['inicio_s']:.1f}s a {row['fim_s']:.1f}s)",
                f"- **Fala:** *\"{row['transcricao']}\"*",
                f"- **Texto indicou:** `{row['emocao_discurso_texto']}` | **Voz indicou:** `{row['emocao_voz_primaria']}`",
                f"- **Nível:** {row['nivel_discordancia']} — **Motivo:** {row['explicacao_clinica']}",
                ""
            ])

    report_md = "\n".join(lines)
    output_md_path.write_text(report_md, encoding="utf-8")
    df_full.to_csv("analise_segmentos_clinicos.csv", index=False, encoding="utf-8-sig")
    return report_md


# ==========================================
# EXECUÇÃO DO PIPELINE COMPLETO
# ==========================================
AUDIO_PATH = Path("/content/START_VIDEO_RECORDING_ky56r.wav")

if not AUDIO_PATH.exists():
    raise FileNotFoundError(f"Arquivo de áudio não encontrado em: {AUDIO_PATH}. Por favor, faça o upload.")

print(f"\n1️⃣ Processando áudio: {AUDIO_PATH.name}...")
display(Audio(str(AUDIO_PATH)))

print("\n2️⃣ Detectando sons não-verbais...")
df_non_verbal = detect_clinical_non_verbal_sounds(AUDIO_PATH)

print("\n3️⃣ Transcrevendo áudio...")
transcribed_segments = transcribe_clinical_audio(AUDIO_PATH)

print("\n4️⃣ Executando diarização...")
diarized_turns = run_real_diarization(AUDIO_PATH)
patient_segments = filter_patient_segments(diarized_turns, transcribed_segments)

print("\n5️⃣ Analisando emoções no tom de voz...")
segments_with_voice_emotion = analyze_voice_emotions(AUDIO_PATH, patient_segments, df_non_verbal)

print("\n6️⃣ Analisando emoções no texto e calculando discordâncias...")
df_completo = run_text_and_discordance_analysis(segments_with_voice_emotion)

print("\n7️⃣ Gerando relatório final...")
relatorio_markdown = generate_clinical_discordance_report(df_completo, df_non_verbal)
display(Markdown(relatorio_markdown))
print("\n✅ Pipeline concluído com sucesso!")


1️⃣ Processando áudio: START_VIDEO_RECORDING_ky56r.wav...



2️⃣ Detectando sons não-verbais...

3️⃣ Transcrevendo áudio...


100%|██████████| 91033/91033 [00:36<00:00, 2505.72frames/s]



4️⃣ Executando diarização...

5️⃣ Analisando emoções no tom de voz...

6️⃣ Analisando emoções no texto e calculando discordâncias...

7️⃣ Gerando relatório final...


# 📋 Relatório Clínico Analítico: Eventos Não-Verbais e Discordâncias Emocionais

## 1. Resumo Quantitativo da Consulta
- **Total de segmentos analisados:** 138
- **Total de discordâncias detectadas:** 81 (58.7%)
- **Total de eventos não-verbais:** 18

## 2. Sons Não-Verbais Detectados
| Início (s) | Fim (s) | Evento Clínico Detectado | Confiança Acústica |
| :---: | :---: | :--- | :---: |
| 78.60s | 79.80s | **Espirro** | 17.2% |
| 100.20s | 101.40s | **Espirro** | 14.1% |
| 103.20s | 104.40s | **Limpeza de Garganta (Pigarro)** | 18.1% |
| 300.60s | 301.80s | **Limpeza de Garganta (Pigarro)** | 21.3% |
| 357.00s | 358.20s | **Espirro** | 32.8% |
| 361.80s | 363.00s | **Limpeza de Garganta (Pigarro)** | 22.1% |
| 630.00s | 631.20s | **Limpeza de Garganta (Pigarro)** | 23.9% |
| 681.60s | 683.40s | **Espirro** | 53.7% |
| 684.00s | 685.80s | **Espirro** | 88.3% |
| 686.40s | 688.20s | **Limpeza de Garganta (Pigarro)** | 55.3% |
| 757.20s | 758.40s | **Espirro** | 12.9% |
| 784.80s | 786.00s | **Espirro** | 14.6% |
| 810.00s | 811.80s | **Ronco / Obstrução Nasal** | 19.2% |
| 817.80s | 819.60s | **Ronco / Obstrução Nasal** | 59.7% |
| 865.20s | 866.40s | **Espirro** | 40.9% |
| 871.20s | 872.40s | **Ronco / Obstrução Nasal** | 14.3% |
| 897.00s | 898.20s | **Espirro** | 26.2% |
| 899.40s | 900.60s | **Ronco / Obstrução Nasal** | 52.6% |

## 3. Comparativo Geral: Texto vs Voz
| Seg # | Tempo | Transcrição | Emoção Texto | Emoção Voz | Sons Próximos | Discordância |
| :---: | :---: | :--- | :--- | :--- | :--- | :---: |
| 5 | 4.0s–7.0s | "I can't hear you, yes." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 8 | 12.0s–13.0s | "I'm ready." | Calma | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 9 | 13.0s–14.0s | "I'm ready when you are." | Calma | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 14 | 24.0s–25.0s | "Hello, Papa." | Alegria | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 15 | 25.0s–26.0s | "Could you just confirm your name?" | Neutro | Calma | — | ✅ Nenhuma |
| 29 | 46.6s–54.6s | "Of course, so my name is Carly Webb and my date of birth is the 12th of July 1967." | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 31 | 63.8s–68.6s | "I'm having quite a bit of discomfort with some shoulder pain." | Desconforto | Calma | — | ⚠️ **Moderada** |
| 32 | 68.6s–76.6s | "So I had a rotate cuff pair in my shoulder and I'm also experiencing quite a bit of weight loss." | Desconforto | Tristeza / Abatimento | Espirro | ✅ Nenhuma |
| 33 | 76.6s–82.2s | "And so I just wanted to ask some questions to you and see how we can get resolved." | Calma | Tristeza / Abatimento | Espirro | ⚠️ **Moderada** |
| 34 | 82.2s–84.2s | "Because disturbing my sleep quite a bit." | Desconforto | Tristeza / Abatimento | Espirro | ✅ Nenhuma |
| 36 | 97.4s–99.4s | "Great. Thank you." | Alegria | Calma | Espirro | ⚠️ **Leve** |
| 37 | 101.4s–103.4s | "How is the pain?" | Neutro | Alegria / Entusiasmo | Espirro, Limpeza de Garganta (Pigarro) | ⚠️ **Leve** |
| 38 | 103.4s–108.2s | "The pain is quite deep and achy pain. So it's not there all the time." | Desconforto | Calma | Espirro, Limpeza de Garganta (Pigarro) | ⚠️ **Moderada** |
| 39 | 108.2s–114.2s | "But especially during the night, I can feel it. It's waking me up every now and again." | Desconforto | Tristeza / Abatimento | — | ✅ Nenhuma |
| 40 | 114.2s–116.2s | "For four times a week." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 41 | 116.2s–120.2s | "Yeah, so it's quite an unpleasant feeling." | Desconforto | Calma | — | ⚠️ **Moderada** |
| 44 | 135.0s–137.4s | "So I sleep on that side generally." | Neutro | Calma | — | ✅ Nenhuma |
| 45 | 137.4s–140.2s | "So it's on my right side and I generally sleep on that side." | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 46 | 140.2s–144.6s | "But I don't know if that's the reason for it waking me up in the evenings." | Ansiedade | Tristeza / Abatimento | — | ✅ Nenhuma |
| 47 | 144.6s–151.8s | "But I do find like everyday tasks like getting dressed or throwing a ball or cutting onions." | Desconforto | Tristeza / Abatimento | — | ✅ Nenhuma |
| 48 | 151.8s–153.8s | "I find that already difficult as well." | Desconforto | Desconforto / Aversão | — | ✅ Nenhuma |
| 49 | 154.2s–158.2s | "So I've started using my left hand and I'm usually right handed." | Desconforto | Alegria / Entusiasmo | — | ⚠️ **Moderada** |
| 50 | 158.2s–163.4s | "But I'm using my left hand now. So I've had to resort to using my other hand." | Desconforto | Tristeza / Abatimento | — | ✅ Nenhuma |
| 52 | 165.4s–167.4s | "Yes." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 54 | 175.0s–175.8s | "During the day." | Neutro | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 56 | 177.8s–182.2s | "During the day, I probably give it maybe a four or five." | Neutro | Calma | — | ✅ Nenhuma |
| 57 | 182.2s–185.8s | "But definitely during the night, probably about eight." | Desconforto | Tristeza / Abatimento | — | ✅ Nenhuma |
| 59 | 196.6s–197.8s | "Yeah." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 64 | 217.4s–219.4s | "That is great. Yes. Thank you." | Alegria | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 67 | 231.0s–233.0s | "No diseases that I can mention." | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 68 | 233.0s–236.2s | "But I do have diagnosis of hypertension." | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 69 | 236.2s–241.0s | "So my blood pressure currently is 133 over 87." | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 70 | 241.0s–246.2s | "And so I've been told that that's high enough to put me on anti-hypertensives." | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 71 | 246.2s–249.8s | "So I'm currently taking medication for that in a minute." | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 75 | 262.6s–264.6s | "So the medication I'm taking is Ramakram." | Neutro | Calma | — | ✅ Nenhuma |
| 76 | 265.4s–267.0s | "But I take it once daily." | Neutro | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 77 | 267.0s–270.6s | "So another thing that I can mention is that I have quite a" | Neutro | Neutro | — | ✅ Nenhuma |
| 78 | 271.8s–277.4s | "sedentary lifestyle is in that I find it difficult to complete physically exerting tasks" | Desconforto | Calma | — | ⚠️ **Moderada** |
| 79 | 278.2s–279.0s | "and things like that." | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 80 | 279.0s–282.2s | "So I was wondering if the two maybe interlinked" | Ansiedade | Calma | — | ⚠️ **Moderada** |
| 81 | 282.2s–286.6s | "in the hypertension diagnosis and my lack of, you know," | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 82 | 286.6s–288.6s | "strenuous activities." | Neutro | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 83 | 288.6s–289.4s | "So to say," | Neutro | Raiva / Irritação | — | ⚠️ **Moderada** |
| 87 | 306.6s–308.2s | "Okay. Okay." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 90 | 318.6s–332.6s | "So when in my early 20s, um, I did, I did like to play a lot of sports, but I feel like," | Tristeza | Tristeza / Abatimento | — | ✅ Nenhuma |
| 91 | 333.4s–338.6s | "definitely, um, you know, this pain that I've had in my shoulder, um," | Desconforto | Tristeza / Abatimento | — | ✅ Nenhuma |
| 92 | 338.6s–344.9s | "I was definitely, you know, exacerbated the lack of my interest in going out of pain," | Desconforto | Tristeza / Abatimento | — | ✅ Nenhuma |
| 93 | 345.5s–348.2s | "any sports or complete any physical exertion." | Desconforto | Tristeza / Abatimento | — | ✅ Nenhuma |
| 94 | 348.2s–353.0s | "So, um, yeah, I definitely think the two are, two are linked." | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 97 | 367.5s–368.0s | "Sorry." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 100 | 372.4s–373.3s | "Yes, please." | Calma | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 101 | 373.3s–374.4s | "Yes, please." | Calma | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 102 | 374.4s–375.4s | "That would be great." | Alegria | Raiva / Irritação | — | ⚠️ **Moderada** |
| 105 | 379.3s–379.9s | "Perfect." | Alegria | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 106 | 379.9s–380.6s | "Perfect." | Alegria | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 107 | 380.6s–381.5s | "What do you have in mind?" | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 111 | 388.9s–391.4s | "At the minute, I'm not taking any pain medication." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 112 | 391.4s–392.2s | "No, no." | Neutro | Calma | — | ✅ Nenhuma |
| 113 | 392.2s–392.8s | "As fingers." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 115 | 399.5s–400.0s | "That's okay." | Calma | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 116 | 400.9s–403.7s | "Um, no, at the minute, I'm not taking any pain medication." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 117 | 404.4s–406.9s | "Um, as I said, I'm trying to just make lifestyle choices." | Calma | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 118 | 407.5s–409.3s | "As I'm just sleeping on the other side." | Calma | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 119 | 409.8s–414.7s | "Um, I'm obviously using my other hand during the day to try and, you know," | Calma | Calma | — | ✅ Nenhuma |
| 120 | 415.2s–418.8s | "minimize the amount of exertion I'm putting on this right shoulder." | Calma | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 121 | 418.8s–420.9s | "So, if there was anything you could suggest," | Neutro | Neutro | — | ✅ Nenhuma |
| 122 | 421.7s–424.3s | "in addition to that, that would be greatly appreciated." | Calma | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 127 | 437.6s–442.4s | "How often would I have to take that?" | Neutro | Calma | — | ✅ Nenhuma |
| 129 | 455.6s–456.0s | "Okay." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 130 | 456.0s–457.2s | "I saw that on the way." | Neutro | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 131 | 457.2s–458.0s | "Okay." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 132 | 458.2s–462.2s | "I'm not allergic to ibuprofencer, so that would be great." | Calma | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 133 | 464.4s–464.7s | "Okay." | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 134 | 467.0s–467.5s | "Great." | Alegria | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 135 | 467.5s–468.4s | "Thank you." | Alegria | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 141 | 485.3s–485.9s | "Perfect." | Alegria | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 142 | 485.9s–487.1s | "Yes, yes, that is my pleasure." | Alegria | Calma | — | ✅ Nenhuma |
| 143 | 487.2s–487.6s | "That is fine." | Calma | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 148 | 502.4s–504.4s | "My mother, yeah, my mother is a live..." | Tristeza | Desconforto / Aversão | — | ✅ Nenhuma |
| 149 | 504.4s–506.1s | "No, my mother is not a live one world." | Tristeza | Desconforto / Aversão | — | ✅ Nenhuma |
| 150 | 506.1s–506.9s | "She passed away." | Tristeza | Raiva / Irritação | — | ✅ Nenhuma |
| 151 | 507.7s–509.2s | "She passed away from breast cancer." | Tristeza | Desconforto / Aversão | — | ✅ Nenhuma |
| 152 | 510.6s–512.8s | "And she used to have hypothyroidism." | Neutro | Neutro | — | ✅ Nenhuma |
| 153 | 514.4s–516.3s | "I don't know what that is, but" | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 155 | 521.1s–521.5s | "Thank you." | Alegria | Raiva / Irritação | — | ⚠️ **Moderada** |
| 158 | 530.3s–533.7s | "He's alive, but he has type two diabetes." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 159 | 535.2s–542.4s | "So, yeah, so both my parents, they had their own problems, but yeah, my father is alive and well." | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 160 | 543.4s–545.4s | "As I said, type two diabetes." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 161 | 545.4s–551.6s | "So, he's quite a large man, not like myself, but yeah, he..." | Neutro | Calma | — | ✅ Nenhuma |
| 162 | 553.1s–555.2s | "I mean, another thing is that I'm losing weight." | Desconforto | Alegria / Entusiasmo | — | ⚠️ **Moderada** |
| 165 | 566.1s–567.2s | "Yes, yes." | Neutro | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 166 | 568.5s–572.3s | "I was just adding that in because talking about my father," | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 167 | 572.3s–573.2s | "it's quite a large man." | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 168 | 574.0s–577.8s | "I thought it could dimension that I'm losing quite a lot of weight." | Desconforto | Tristeza / Abatimento | — | ✅ Nenhuma |
| 169 | 577.8s–582.0s | "And obviously, because I don't do a lot of physical activity," | Desconforto | Tristeza / Abatimento | — | ✅ Nenhuma |
| 170 | 582.0s–583.2s | "I was wondering why that is." | Ansiedade | Tristeza / Abatimento | — | ✅ Nenhuma |
| 171 | 584.8s–589.5s | "I've lost about four and a half kilos just over the last couple of weeks." | Desconforto | Tristeza / Abatimento | — | ✅ Nenhuma |
| 173 | 599.0s–601.3s | "It's affecting me quite a lot at work as well." | Desconforto | Tristeza / Abatimento | — | ✅ Nenhuma |
| 175 | 606.8s–608.0s | "Okay, okay." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 177 | 620.9s–622.4s | "Brilliant, brilliant, brilliant." | Alegria | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 178 | 622.4s–624.1s | "Okay, what should we focus on now?" | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 180 | 631.2s–635.2s | "Yes, so uncertain, so I feel like I'm quite stressed at work." | Ansiedade | Tristeza / Abatimento | Limpeza de Garganta (Pigarro) | ✅ Nenhuma |
| 181 | 635.2s–638.2s | "And so, I don't know if that has something to do with weight loss." | Ansiedade | Neutro | — | ⚠️ **Moderada** |
| 184 | 645.6s–647.2s | "I'm a school teacher." | Neutro | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 185 | 647.8s–651.2s | "So, obviously, it's like, the hour is at nine o'clock to five o'clock." | Neutro | Calma | — | ✅ Nenhuma |
| 186 | 652.8s–656.0s | "And as a teacher, it's very difficult to not go into work." | Desconforto | Tristeza / Abatimento | — | ✅ Nenhuma |
| 187 | 656.8s–660.0s | "So, together with the weight loss and the pain in my arm," | Desconforto | Neutro | — | ⚠️ **Moderada** |
| 188 | 661.6s–664.0s | "as you say, providing anti-inflammatory." | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 189 | 666.4s–671.6s | "Yes, so that's, I'm still trying to work as often as I can." | Calma | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 193 | 697.2s–698.8s | "I don't, I don't see it." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 196 | 700.5s–702.0s | "Thank you, thank you very much." | Alegria | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 197 | 702.5s–705.9s | "I don't smoke at the minute, but I do drink." | Neutro | Calma | — | ✅ Nenhuma |
| 198 | 707.3s–709.2s | "So, I drink wine during the weekend." | Neutro | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 199 | 710.1s–712.4s | "And then I drink a little bit more of the weekends." | Neutro | Calma | — | ✅ Nenhuma |
| 200 | 712.7s–715.9s | "But as I mentioned, I don't smoke." | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 204 | 749.5s–756.3s | "That is not good to hear, but is there a way that we could try and address this?" | Desconforto | Tristeza / Abatimento | Espirro | ✅ Nenhuma |
| 209 | 782.1s–783.2s | "That sounds good." | Alegria | Desconforto / Aversão | Espirro | ⚠️ **Moderada** |
| 210 | 783.2s–787.6s | "That sounds like a good first step, because obviously I don't want this to" | Ansiedade | Calma | Espirro | ⚠️ **Moderada** |
| 211 | 788.5s–790.0s | "progress too much." | Ansiedade | Alegria / Entusiasmo | — | ⚠️ **Moderada** |
| 212 | 791.1s–793.5s | "As I said, my mom passed away from breast cancer." | Tristeza | Alegria / Entusiasmo | — | ⚠️ **Moderada** |
| 213 | 793.5s–794.4s | "Yes." | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 214 | 794.4s–798.3s | "If we can catch it early on, that would be really good." | Calma | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 216 | 808.4s–809.6s | "It's almost like we're both doctors." | Alegria | Desconforto / Aversão | Ronco / Obstrução Nasal | ⚠️ **Moderada** |
| 217 | 811.2s–812.3s | "Yes, great." | Alegria | Desconforto / Aversão | Ronco / Obstrução Nasal | ⚠️ **Moderada** |
| 218 | 813.5s–814.0s | "Almost." | Alegria | Tristeza / Abatimento | Ronco / Obstrução Nasal | ⚠️ **Moderada** |
| 221 | 827.2s–831.6s | "How would we get started with the breast cancer steps?" | Neutro | Desconforto / Aversão | — | ⚠️ **Moderada** |
| 222 | 831.6s–835.3s | "What would be the first thing you're going to refer me and what happens after that?" | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 224 | 850.6s–852.2s | "Okay." | Neutro | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 225 | 852.2s–854.2s | "Okay." | Neutro | Raiva / Irritação | — | ⚠️ **Moderada** |
| 227 | 861.0s–863.2s | "Brilliant." | Alegria | Tristeza / Abatimento | Espirro | ⚠️ **Moderada** |
| 228 | 863.2s–865.5s | "That sounds like a good first step." | Alegria | Desconforto / Aversão | Espirro | ⚠️ **Moderada** |
| 230 | 868.3s–869.5s | "Thank you." | Alegria | Desconforto / Aversão | Espirro, Ronco / Obstrução Nasal | ⚠️ **Moderada** |
| 231 | 869.5s–871.0s | "Thank you very much." | Alegria | Desconforto / Aversão | Ronco / Obstrução Nasal | ⚠️ **Moderada** |
| 233 | 881.1s–881.5s | "Thank you." | Alegria | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 234 | 881.5s–883.2s | "I hope so too." | Calma | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 235 | 883.7s–888.2s | "But that was a good start in terms of addressing all my problems." | Alegria | Tristeza / Abatimento | — | ⚠️ **Moderada** |
| 236 | 889.5s–890.0s | "Thank you." | Alegria | Alegria / Entusiasmo | — | ✅ Nenhuma |
| 237 | 891.3s–893.2s | "I'm pretty wrapped." | Alegria | Tristeza / Abatimento | — | ⚠️ **Moderada** |

## 4. Detalhamento Clínico Programático
### 🔍 Segmento #5 (4.0s a 7.0s)
- **Fala:** *"I can't hear you, yes."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #8 (12.0s a 13.0s)
- **Fala:** *"I'm ready."*
- **Texto indicou:** `Calma` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Calma' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #9 (13.0s a 14.0s)
- **Fala:** *"I'm ready when you are."*
- **Texto indicou:** `Calma` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Calma' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #14 (24.0s a 25.0s)
- **Fala:** *"Hello, Papa."*
- **Texto indicou:** `Alegria` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Alegria' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #29 (46.6s a 54.6s)
- **Fala:** *"Of course, so my name is Carly Webb and my date of birth is the 12th of July 1967."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #31 (63.8s a 68.6s)
- **Fala:** *"I'm having quite a bit of discomfort with some shoulder pain."*
- **Texto indicou:** `Desconforto` | **Voz indicou:** `Calma`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Desconforto' mas Voz indicou 'Calma'.

### 🔍 Segmento #33 (76.6s a 82.2s)
- **Fala:** *"And so I just wanted to ask some questions to you and see how we can get resolved."*
- **Texto indicou:** `Calma` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Calma' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #36 (97.4s a 99.4s)
- **Fala:** *"Great. Thank you."*
- **Texto indicou:** `Alegria` | **Voz indicou:** `Calma`
- **Nível:** Leve — **Motivo:** Sinais físicos (Espirro) contradizem o relato textual positivo/neutro.

### 🔍 Segmento #37 (101.4s a 103.4s)
- **Fala:** *"How is the pain?"*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Alegria / Entusiasmo`
- **Nível:** Leve — **Motivo:** Sinais físicos (Espirro, Limpeza de Garganta (Pigarro)) contradizem o relato textual positivo/neutro.

### 🔍 Segmento #38 (103.4s a 108.2s)
- **Fala:** *"The pain is quite deep and achy pain. So it's not there all the time."*
- **Texto indicou:** `Desconforto` | **Voz indicou:** `Calma`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Desconforto' mas Voz indicou 'Calma'.

### 🔍 Segmento #40 (114.2s a 116.2s)
- **Fala:** *"For four times a week."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #41 (116.2s a 120.2s)
- **Fala:** *"Yeah, so it's quite an unpleasant feeling."*
- **Texto indicou:** `Desconforto` | **Voz indicou:** `Calma`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Desconforto' mas Voz indicou 'Calma'.

### 🔍 Segmento #45 (137.4s a 140.2s)
- **Fala:** *"So it's on my right side and I generally sleep on that side."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #49 (154.2s a 158.2s)
- **Fala:** *"So I've started using my left hand and I'm usually right handed."*
- **Texto indicou:** `Desconforto` | **Voz indicou:** `Alegria / Entusiasmo`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Desconforto' mas Voz indicou 'Alegria / Entusiasmo'.

### 🔍 Segmento #52 (165.4s a 167.4s)
- **Fala:** *"Yes."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #59 (196.6s a 197.8s)
- **Fala:** *"Yeah."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #67 (231.0s a 233.0s)
- **Fala:** *"No diseases that I can mention."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #68 (233.0s a 236.2s)
- **Fala:** *"But I do have diagnosis of hypertension."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #69 (236.2s a 241.0s)
- **Fala:** *"So my blood pressure currently is 133 over 87."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #70 (241.0s a 246.2s)
- **Fala:** *"And so I've been told that that's high enough to put me on anti-hypertensives."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #71 (246.2s a 249.8s)
- **Fala:** *"So I'm currently taking medication for that in a minute."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #78 (271.8s a 277.4s)
- **Fala:** *"sedentary lifestyle is in that I find it difficult to complete physically exerting tasks"*
- **Texto indicou:** `Desconforto` | **Voz indicou:** `Calma`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Desconforto' mas Voz indicou 'Calma'.

### 🔍 Segmento #79 (278.2s a 279.0s)
- **Fala:** *"and things like that."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #80 (279.0s a 282.2s)
- **Fala:** *"So I was wondering if the two maybe interlinked"*
- **Texto indicou:** `Ansiedade` | **Voz indicou:** `Calma`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Ansiedade' mas Voz indicou 'Calma'.

### 🔍 Segmento #81 (282.2s a 286.6s)
- **Fala:** *"in the hypertension diagnosis and my lack of, you know,"*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #83 (288.6s a 289.4s)
- **Fala:** *"So to say,"*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Raiva / Irritação`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Raiva / Irritação'.

### 🔍 Segmento #87 (306.6s a 308.2s)
- **Fala:** *"Okay. Okay."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #94 (348.2s a 353.0s)
- **Fala:** *"So, um, yeah, I definitely think the two are, two are linked."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #97 (367.5s a 368.0s)
- **Fala:** *"Sorry."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #101 (373.3s a 374.4s)
- **Fala:** *"Yes, please."*
- **Texto indicou:** `Calma` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Calma' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #102 (374.4s a 375.4s)
- **Fala:** *"That would be great."*
- **Texto indicou:** `Alegria` | **Voz indicou:** `Raiva / Irritação`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Alegria' mas Voz indicou 'Raiva / Irritação'.

### 🔍 Segmento #107 (380.6s a 381.5s)
- **Fala:** *"What do you have in mind?"*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #111 (388.9s a 391.4s)
- **Fala:** *"At the minute, I'm not taking any pain medication."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #113 (392.2s a 392.8s)
- **Fala:** *"As fingers."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #115 (399.5s a 400.0s)
- **Fala:** *"That's okay."*
- **Texto indicou:** `Calma` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Calma' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #116 (400.9s a 403.7s)
- **Fala:** *"Um, no, at the minute, I'm not taking any pain medication."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #117 (404.4s a 406.9s)
- **Fala:** *"Um, as I said, I'm trying to just make lifestyle choices."*
- **Texto indicou:** `Calma` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Calma' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #118 (407.5s a 409.3s)
- **Fala:** *"As I'm just sleeping on the other side."*
- **Texto indicou:** `Calma` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Calma' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #120 (415.2s a 418.8s)
- **Fala:** *"minimize the amount of exertion I'm putting on this right shoulder."*
- **Texto indicou:** `Calma` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Calma' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #122 (421.7s a 424.3s)
- **Fala:** *"in addition to that, that would be greatly appreciated."*
- **Texto indicou:** `Calma` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Calma' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #129 (455.6s a 456.0s)
- **Fala:** *"Okay."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #131 (457.2s a 458.0s)
- **Fala:** *"Okay."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #132 (458.2s a 462.2s)
- **Fala:** *"I'm not allergic to ibuprofencer, so that would be great."*
- **Texto indicou:** `Calma` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Calma' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #133 (464.4s a 464.7s)
- **Fala:** *"Okay."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #153 (514.4s a 516.3s)
- **Fala:** *"I don't know what that is, but"*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #155 (521.1s a 521.5s)
- **Fala:** *"Thank you."*
- **Texto indicou:** `Alegria` | **Voz indicou:** `Raiva / Irritação`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Alegria' mas Voz indicou 'Raiva / Irritação'.

### 🔍 Segmento #158 (530.3s a 533.7s)
- **Fala:** *"He's alive, but he has type two diabetes."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #159 (535.2s a 542.4s)
- **Fala:** *"So, yeah, so both my parents, they had their own problems, but yeah, my father is alive and well."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #160 (543.4s a 545.4s)
- **Fala:** *"As I said, type two diabetes."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #162 (553.1s a 555.2s)
- **Fala:** *"I mean, another thing is that I'm losing weight."*
- **Texto indicou:** `Desconforto` | **Voz indicou:** `Alegria / Entusiasmo`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Desconforto' mas Voz indicou 'Alegria / Entusiasmo'.

### 🔍 Segmento #166 (568.5s a 572.3s)
- **Fala:** *"I was just adding that in because talking about my father,"*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #167 (572.3s a 573.2s)
- **Fala:** *"it's quite a large man."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #175 (606.8s a 608.0s)
- **Fala:** *"Okay, okay."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #178 (622.4s a 624.1s)
- **Fala:** *"Okay, what should we focus on now?"*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #181 (635.2s a 638.2s)
- **Fala:** *"And so, I don't know if that has something to do with weight loss."*
- **Texto indicou:** `Ansiedade` | **Voz indicou:** `Neutro`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Ansiedade' mas Voz indicou 'Neutro'.

### 🔍 Segmento #187 (656.8s a 660.0s)
- **Fala:** *"So, together with the weight loss and the pain in my arm,"*
- **Texto indicou:** `Desconforto` | **Voz indicou:** `Neutro`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Desconforto' mas Voz indicou 'Neutro'.

### 🔍 Segmento #188 (661.6s a 664.0s)
- **Fala:** *"as you say, providing anti-inflammatory."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #189 (666.4s a 671.6s)
- **Fala:** *"Yes, so that's, I'm still trying to work as often as I can."*
- **Texto indicou:** `Calma` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Calma' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #193 (697.2s a 698.8s)
- **Fala:** *"I don't, I don't see it."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #196 (700.5s a 702.0s)
- **Fala:** *"Thank you, thank you very much."*
- **Texto indicou:** `Alegria` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Alegria' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #200 (712.7s a 715.9s)
- **Fala:** *"But as I mentioned, I don't smoke."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #209 (782.1s a 783.2s)
- **Fala:** *"That sounds good."*
- **Texto indicou:** `Alegria` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Alegria' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #210 (783.2s a 787.6s)
- **Fala:** *"That sounds like a good first step, because obviously I don't want this to"*
- **Texto indicou:** `Ansiedade` | **Voz indicou:** `Calma`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Ansiedade' mas Voz indicou 'Calma'.

### 🔍 Segmento #211 (788.5s a 790.0s)
- **Fala:** *"progress too much."*
- **Texto indicou:** `Ansiedade` | **Voz indicou:** `Alegria / Entusiasmo`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Ansiedade' mas Voz indicou 'Alegria / Entusiasmo'.

### 🔍 Segmento #212 (791.1s a 793.5s)
- **Fala:** *"As I said, my mom passed away from breast cancer."*
- **Texto indicou:** `Tristeza` | **Voz indicou:** `Alegria / Entusiasmo`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Tristeza' mas Voz indicou 'Alegria / Entusiasmo'.

### 🔍 Segmento #213 (793.5s a 794.4s)
- **Fala:** *"Yes."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #214 (794.4s a 798.3s)
- **Fala:** *"If we can catch it early on, that would be really good."*
- **Texto indicou:** `Calma` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Calma' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #216 (808.4s a 809.6s)
- **Fala:** *"It's almost like we're both doctors."*
- **Texto indicou:** `Alegria` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Alegria' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #217 (811.2s a 812.3s)
- **Fala:** *"Yes, great."*
- **Texto indicou:** `Alegria` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Alegria' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #218 (813.5s a 814.0s)
- **Fala:** *"Almost."*
- **Texto indicou:** `Alegria` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Alegria' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #221 (827.2s a 831.6s)
- **Fala:** *"How would we get started with the breast cancer steps?"*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #222 (831.6s a 835.3s)
- **Fala:** *"What would be the first thing you're going to refer me and what happens after that?"*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #224 (850.6s a 852.2s)
- **Fala:** *"Okay."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #225 (852.2s a 854.2s)
- **Fala:** *"Okay."*
- **Texto indicou:** `Neutro` | **Voz indicou:** `Raiva / Irritação`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Neutro' mas Voz indicou 'Raiva / Irritação'.

### 🔍 Segmento #227 (861.0s a 863.2s)
- **Fala:** *"Brilliant."*
- **Texto indicou:** `Alegria` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Alegria' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #228 (863.2s a 865.5s)
- **Fala:** *"That sounds like a good first step."*
- **Texto indicou:** `Alegria` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Alegria' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #230 (868.3s a 869.5s)
- **Fala:** *"Thank you."*
- **Texto indicou:** `Alegria` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Alegria' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #231 (869.5s a 871.0s)
- **Fala:** *"Thank you very much."*
- **Texto indicou:** `Alegria` | **Voz indicou:** `Desconforto / Aversão`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Alegria' mas Voz indicou 'Desconforto / Aversão'.

### 🔍 Segmento #234 (881.5s a 883.2s)
- **Fala:** *"I hope so too."*
- **Texto indicou:** `Calma` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Calma' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #235 (883.7s a 888.2s)
- **Fala:** *"But that was a good start in terms of addressing all my problems."*
- **Texto indicou:** `Alegria` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Alegria' mas Voz indicou 'Tristeza / Abatimento'.

### 🔍 Segmento #237 (891.3s a 893.2s)
- **Fala:** *"I'm pretty wrapped."*
- **Texto indicou:** `Alegria` | **Voz indicou:** `Tristeza / Abatimento`
- **Nível:** Moderada — **Motivo:** Divergência direta: Texto indicou 'Alegria' mas Voz indicou 'Tristeza / Abatimento'.



✅ Pipeline concluído com sucesso!
